In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
print(os.getcwd())

In [ ]:
phase_6_kubernetes

In [ ]:
!ls

In [ ]:
%%writefile Dockerfile
# =========================================================
# ETAPA 1: Construcción y dependencias
# =========================================================
FROM python:3.11-slim AS builder

# Evita que Python escriba archivos .pyc en el disco y asegura salida en tiempo real
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /build

# Instalar dependencias del sistema necesarias para compilar ciertas ruedas de Python si fuera necesario
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Crear entorno virtual aislado para copiarlo limpiamente en la siguiente etapa
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Copiar e instalar dependencias (Boto3)
# Si no tienes un requirements.txt, lo instalamos directamente para simplificar tu flujo
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir boto3

# =========================================================
# ETAPA 2: Imagen de ejecución final (Producción / K8s)
# =========================================================
FROM python:3.11-slim AS runner

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PATH="/opt/venv/bin:$PATH"

WORKDIR /app

# Copiar el entorno virtual con las librerías pre-instaladas desde la etapa builder
COPY --from=builder /opt/venv /opt/venv

# Copiar únicamente el código de nuestra aplicación
COPY app.py .

# Crear las carpetas locales de auditoría que usa el script y asignar permisos
RUN mkdir -p data/archive/input data/archive/output && \
    chmod -R 777 data

# Configuración de variables de entorno por defecto (Valores base de resguardo)
# En Kubernetes, estas variables serán sobreescritas por el ConfigMap o el manifiesto del CronJob
ENV SIMULACION="True" \
    AWS_ENDPOINT_URL="http://localstack-service:4566" \
    S3_BUCKET_NAME="fase5-storage-bucket" \
    AWS_INPUT_KEY="raw_telemetry.csv" \
    DYNAMODB_LOCK_TABLE="fase5-execution-locks" \
    AWS_DEFAULT_REGION="us-east-1"

# Crear un usuario sin privilegios (Práctica obligatoria de seguridad para Kubernetes)
# Evita ejecutar contenedores como root en el clúster
RUN useradd -u 8888 appuser && chown -R appuser:appuser /app
USER appuser

# Comando de ejecución único para tareas por lotes (K8s Jobs/CronJobs)
CMD ["python", "app.py"]

In [1]:
!cd /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container

# !docker build -t architect-python-batch:v6 .

In [3]:
!docker build -t architect-python-batch:v6 .

[+] Building 0.0s (0/1)                                                         
[+] Building 0.1s (1/2)                                                         
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 37B                                        0.0s
[+] Building 0.2s (2/3)                                                         
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 37B                                        0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 2B                                            0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.0s
[+] Building 0.4s (2/3)                                                         
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 37B 

In [ ]:
# =========================================================
# 1. CONFIGMAP - CENTRALIZACIÓN DE VARIABLES DE ENTORNO
# =========================================================
apiVersion: v1
kind: ConfigMap
metadata:
  name: batch-pipeline-config
  namespace: default
data:
  SIMULACION: "False"  # Cambiado a False para interactuar directamente con LocalStack
  AWS_ENDPOINT_URL: "http://localstack-service:4566"  # DNS interno del Service de LocalStack en el clúster
  S3_BUCKET_NAME: "fase5-storage-bucket"
  AWS_INPUT_KEY: "raw_telemetry.csv"
  DYNAMODB_LOCK_TABLE: "fase5-execution-locks"
  AWS_DEFAULT_REGION: "us-east-1"
---
# =========================================================
# 2. CRONJOB - ORQUESTACIÓN DE LA EJECUCIÓN POR LOTES
# =========================================================
apiVersion: batch/v1
kind: CronJob
metadata:
  name: architect-python-batch-job
  namespace: default
spec:
  # Programación: Se ejecuta cada minuto (* * * * *)
  schedule: "*/1 * * * *"
  
  # Estrategias de concurrencia y retención de historial
  concurrencyPolicy: Forbid            # Si el lote anterior se retrasa, prohíbe abrir un Pod nuevo en paralelo
  successfulJobsHistoryLimit: 3        # Mantiene el historial/logs de las últimas 3 ejecuciones exitosas
  failedJobsHistoryLimit: 5            # Mantiene el historial de las últimas 5 fallidas para depuración
  startingDeadlineSeconds: 60          # Tiempo límite para iniciar el Job si el planificador se retrasa
  
  jobTemplate:
    spec:
      template:
        metadata:
          labels:
            app: telemetry-batch-processor
        spec:
          # Inyección de todas las variables del ConfigMap al contenedor
          containers:
          - name: batch-processor
            image: architect-python-batch:v6
            imagePullPolicy: IfNotPresent  # Utiliza la imagen local construida si ya se encuentra en el nodo
            
            # Carga masiva de la configuración dinámica
            envFrom:
            - configMapRef:
                name: batch-pipeline-config
            
            # Forzar límites de recursos para evitar fugas de memoria en el clúster
            resources:
              requests:
                memory: "64Mi"
                cpu: "100m"
              limits:
                memory: "128Mi"
                cpu: "200m"

          # Reinicio: En tareas Batch/CronJob de K8s, Never o OnFailure son obligatorios
          restartPolicy: OnFailure

In [ ]:
# Practice and review, practica y repaso miercoles taller
name: batch-pipeline-configMapRef        Configmap     ...viene de 1 
name: architect-python-batch-jobTemplate  Cronjob    ...viene de 2
    jobTemplate:
        spec:
      template:
        metadata:
          labels:
            app: telemetry-batch-processor
        spec:
          # Inyección de todas las variables del ConfigMap al contenedor
          containers:
          - name: batch-processor
            image: architect-python-batch:v6
            imagePullPolicy: IfNotPresent  # Utiliza la imagen local construida si ya se encuentra en el nodo
            
            # Carga masiva de la configuración dinámica
            envFrom:
            - configMapRef:
                name: batch-pipeline-config

In [2]:
%%writefile docker-compose.yml
# =========================================================
# 1. CONFIGMAP - CENTRALIZACIÓN DE VARIABLES DE ENTORNO
# =========================================================
apiVersion: v1
kind: ConfigMap
metadata:
  name: batch-pipeline-config
  namespace: default
data:
  SIMULACION: "False"  # Cambiado a False para interactuar directamente con LocalStack
  AWS_ENDPOINT_URL: "http://localstack-service:4566"  # DNS interno del Service de LocalStack en el clúster
  S3_BUCKET_NAME: "fase5-storage-bucket"
  AWS_INPUT_KEY: "raw_telemetry.csv"
  DYNAMODB_LOCK_TABLE: "fase5-execution-locks"
  AWS_DEFAULT_REGION: "us-east-1"
---
# =========================================================
# 2. CRONJOB - ORQUESTACIÓN DE LA EJECUCIÓN POR LOTES
# =========================================================
apiVersion: batch/v1
kind: CronJob
metadata:
  name: architect-python-batch-job
  namespace: default
spec:
  # Programación: Se ejecuta cada minuto (* * * * *)
  schedule: "*/1 * * * *"
  
  # Estrategias de concurrencia y retención de historial
  concurrencyPolicy: Forbid            # Si el lote anterior se retrasa, prohíbe abrir un Pod nuevo en paralelo
  successfulJobsHistoryLimit: 3        # Mantiene el historial/logs de las últimas 3 ejecuciones exitosas
  failedJobsHistoryLimit: 5            # Mantiene el historial de las últimas 5 fallidas para depuración
  startingDeadlineSeconds: 60          # Tiempo límite para iniciar el Job si el planificador se retrasa
  
  jobTemplate:
    spec:
      template:
        metadata:
          labels:
            app: telemetry-batch-processor
        spec:
          # Inyección de todas las variables del ConfigMap al contenedor
          containers:
          - name: batch-processor
            image: architect-python-batch:v6
            imagePullPolicy: IfNotPresent  # Utiliza la imagen local construida si ya se encuentra en el nodo
            
            # Carga masiva de la configuración dinámica
            envFrom:
            - configMapRef:
                name: batch-pipeline-config
            
            # Forzar límites de recursos para evitar fugas de memoria en el clúster
            resources:
              requests:
                memory: "64Mi"
                cpu: "100m"
              limits:
                memory: "128Mi"
                cpu: "200m"

          # Reinicio: En tareas Batch/CronJob de K8s, Never o OnFailure son obligatorios
          restartPolicy: OnFailure

Writing docker-compose.yml


In [7]:
%%writefile cronjob.yaml
# =========================================================
# 1. CONFIGMAP - CENTRALIZACIÓN DE VARIABLES DE ENTORNO
# =========================================================
apiVersion: v1
kind: ConfigMap
metadata:
  name: batch-pipeline-config
  namespace: default
data:
  SIMULACION: "False"  # Cambiado a False para interactuar directamente con LocalStack
  AWS_ENDPOINT_URL: "http://localstack-service:4566"  # DNS interno del Service de LocalStack en el clúster
  S3_BUCKET_NAME: "fase5-storage-bucket"
  AWS_INPUT_KEY: "raw_telemetry.csv"
  DYNAMODB_LOCK_TABLE: "fase5-execution-locks"
  AWS_DEFAULT_REGION: "us-east-1"
---
# =========================================================
# 2. CRONJOB - ORQUESTACIÓN DE LA EJECUCIÓN POR LOTES
# =========================================================
apiVersion: batch/v1
kind: CronJob
metadata:
  name: architect-python-batch-job
  namespace: default
spec:
  # Programación: Se ejecuta cada minuto (* * * * *)
  schedule: "*/1 * * * *"
  
  # Estrategias de concurrencia y retención de historial
  concurrencyPolicy: Forbid            # Si el lote anterior se retrasa, prohíbe abrir un Pod nuevo en paralelo
  successfulJobsHistoryLimit: 3        # Mantiene el historial/logs de las últimas 3 ejecuciones exitosas
  failedJobsHistoryLimit: 5            # Mantiene el historial de las últimas 5 fallidas para depuración
  startingDeadlineSeconds: 60          # Tiempo límite para iniciar el Job si el planificador se retrasa
  
  jobTemplate:
    spec:
      template:
        metadata:
          labels:
            app: telemetry-batch-processor
        spec:
          # Inyección de todas las variables del ConfigMap al contenedor
          containers:
          - name: batch-processor
            image: architect-python-batch:v6
            imagePullPolicy: IfNotPresent  # Utiliza la imagen local construida si ya se encuentra en el nodo
            
            # Carga masiva de la configuración dinámica
            envFrom:
            - configMapRef:
                name: batch-pipeline-config
            
            # Forzar límites de recursos para evitar fugas de memoria en el clúster
            resources:
              requests:
                memory: "64Mi"
                cpu: "100m"
              limits:
                memory: "128Mi"
                cpu: "200m"

          # Reinicio: En tareas Batch/CronJob de K8s, Never o OnFailure son obligatorios
          restartPolicy: OnFailure

Writing cronjob.yaml


In [8]:
# 1. Aplicar la configuración al clúster
!kubectl apply -f cronjob.yaml



The connection to the server localhost:8080 was refused - did you specify the right host or port?


In [10]:
!kubectl config use-context docker-desktop

error: no context exists with the name: "docker-desktop"


In [12]:
!minikube start

/bin/bash: minikube: command not found


In [13]:
!kubectl config use-context docker-desktop

Switched to context "docker-desktop".


In [14]:
!kubectl cluster-info

Kubernetes control plane is running at https://kubernetes.docker.internal:6443
CoreDNS is running at https://kubernetes.docker.internal:6443/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.


In [ ]:
# 2. Verificar que el CronJob esté registrado y programado
kubectl get cronjob



In [ ]:
# 3. Monitorear los Pods efímeros que se crean minuto a minuto
kubectl get pods --watch



In [ ]:
# 4. Revisar los logs del contenedor para verificar la salida de auditoría y LocalStack
kubectl logs -l app=telemetry-batch-processor --tail=50

In [ ]:
que bien, ahora lo que sigue es los siguientes comandos: # 1. Aplicar la configuración al clúster
kubectl apply -f cronjob.yaml

# 4. Revisar los logs del contenedor para verificar la salida de auditoría y LocalStack
kubectl logs -l app=telemetry-batch-processor --tail=50, pero cuando corri el primero este fue el output: The connection to the server localhost:8080 was refused - did you specify the right host or port? 


In [9]:
!docker ps

CONTAINER ID   IMAGE                         COMMAND                  CREATED       STATUS                 PORTS                                             NAMES
2ddcfe6bdf5f   localstack/localstack:2.3.2   "docker-entrypoint.sh"   7 hours ago   Up 7 hours (healthy)   4510-4559/tcp, 5678/tcp, 0.0.0.0:4566->4566/tcp   localstack_simulador


In [15]:
#### Continuacion desde: !kubectl cluster-info
!kubectl apply -f cronjob.yaml

configmap/batch-pipeline-config created
cronjob.batch/architect-python-batch-job created


In [16]:
!kubectl get cronjob

NAME                         SCHEDULE      SUSPEND   ACTIVE   LAST SCHEDULE   AGE
architect-python-batch-job   */1 * * * *   False     1        16s             47s


In [17]:
!kubectl get pods --watch

NAME                                        READY   STATUS    RESTARTS      AGE
architect-python-batch-job-29718950-kbzjq   1/1     Running   4 (59s ago)   2m25s
architect-python-batch-job-29718950-kbzjq   0/1     Error     4 (60s ago)   2m26s
architect-python-batch-job-29718950-kbzjq   0/1     CrashLoopBackOff   4 (12s ago)   2m38s
^C


In [18]:
!kubectl logs architect-python-batch-job-29718950-kbzjq

🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
🎛️  Modo de Simulación: DESACTIVADO (1,000 filas reales en LocalStack S3)


In [19]:
!kubectl logs architect-python-batch-job-29718950-kbzjq

Error from server (NotFound): pods "architect-python-batch-job-29718950-kbzjq" not found


In [20]:
!kubectl get pods

NAME                                        READY   STATUS             RESTARTS      AGE
architect-python-batch-job-29718956-d8n2h   0/1     CrashLoopBackOff   5 (81s ago)   5m21s


In [21]:
!kubectl logs architect-python-batch-job-29718956-d8n2h

Error from server (NotFound): pods "architect-python-batch-job-29718956-d8n2h" not found


In [22]:
!kubectl patch cronjob architect-python-batch-job -p '{"spec" : {"suspend" : true }}'

cronjob.batch/architect-python-batch-job patched


In [23]:
!kubectl create job my-test-job --from=cronjob/architect-python-batch-job

job.batch/my-test-job created


In [24]:
!kubectl get pods

NAME                                        READY   STATUS             RESTARTS      AGE
architect-python-batch-job-29718963-mm2vx   0/1     CrashLoopBackOff   5 (36s ago)   4m34s
my-test-job-pxrcx                           0/1     Error              1 (11s ago)   27s


In [ ]:
NAME                                        READY   STATUS             RESTARTS      AGE
architect-python-batch-job-29718963-mm2vx   0/1     CrashLoopBackOff   5 (36s ago)   4m34s
my-test-job-pxrcx                           0/1     Error              1 (11s ago)   27s

In [25]:
!kubectl logs

error: expected 'logs [-f] [-p] (POD | TYPE/NAME) [-c CONTAINER]'.
POD or TYPE/NAME is a required argument for the logs command
See 'kubectl logs -h' for help and examples


In [26]:
!kubectl logs -l job-name=my-test-job --tail=100

🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
🎛️  Modo de Simulación: DESACTIVADO (1,000 filas reales en LocalStack S3)
⏱️ Iniciando ejecución única de Kubernetes CronJob para el bloque: cron_batch_20260704_0410
Traceback (most recent call last):
  File "/app/app.py", line 172, in <module>
    if acquire_lock(dynamodb_client, execution_id):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/app/app.py", line 37, in acquire_lock
    dynamodb_client.put_item(
  File "/opt/venv/lib/python3.11/site-packages/botocore/client.py", line 606, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.11/site-packages/botocore/client.py", line 1076, in _make_api_call
    http, parsed_response = self._make_request(
                    

In [27]:
!kubectl delete job my-test-job

job.batch "my-test-job" deleted


In [28]:
!kubectl apply -f cronjob.yaml 

cronjob.batch/architect-python-batch-job configured


In [29]:
!kubectl patch cronjob architect-python-batch-job -p '{"spec" : {"suspend" : false }}'

cronjob.batch/architect-python-batch-job patched


In [30]:
!kubectl get pods --watch

NAME                                        READY   STATUS    RESTARTS   AGE
architect-python-batch-job-29718976-kb4jt   1/1     Running   0          76s
^C


In [31]:
!kubectl logs architect-python-batch-job-29718976-kb4jt --tail=50


LocalStack version: 2.3.2
LocalStack build date: 2023-10-03
LocalStack build git hash: 1bccf790

2026-07-04T04:16:25.707  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T04:16:25.707  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T04:16:26.004  INFO --- [  MainThread] localstack.utils.bootstrap : Execution of "start_runtime_components" took 1207.61ms
Ready.


In [ ]:
# ... (parte superior de tu archivo igual)
        spec:
          containers:
          - name: telemetry-batch-processor
            # === MODIFICA ESTA LÍNEA CON TU IMAGEN REAL DE PYTHON ===
            image: telemetry-batch-processor:latest 
            imagePullPolicy: IfNotPresent
            env:
            - name: AWS_ACCESS_KEY_ID
              value: "mock_key"
            - name: AWS_SECRET_ACCESS_KEY
              value: "mock_secret"
            - name: AWS_DEFAULT_REGION
              value: "us-east-1"
            - name: AWS_ENDPOINT_URL
              value: "http://host.docker.internal:4566"
            - name: SIMULATION_MODE
              value: "OFF"
          restartPolicy: OnFailure

In [32]:
!kubectl delete pod architect-python-batch-job-29718976-kb4jt

pod "architect-python-batch-job-29718976-kb4jt" deleted


In [33]:
!kubectl apply -f cronjob.yaml

cronjob.batch/architect-python-batch-job configured


In [34]:
!kubectl get pods --watch

NAME                                        READY   STATUS    RESTARTS   AGE
architect-python-batch-job-29718976-vxqvb   1/1     Running   0          90s
^C


In [35]:
!kubectl logs <architect-python-batch-job-29718976-vxqvb>

/bin/bash: -c: line 0: syntax error near unexpected token `newline'
/bin/bash: -c: line 0: `kubectl logs <architect-python-batch-job-29718976-vxqvb>'


In [36]:
!kubectl logs architect-python-batch-job-29718976-vxqvb


LocalStack version: 2.3.2
LocalStack build date: 2023-10-03
LocalStack build git hash: 1bccf790

2026-07-04T04:26:39.283  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T04:26:39.283  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T04:26:39.506  INFO --- [  MainThread] localstack.utils.bootstrap : Execution of "start_runtime_components" took 1237.69ms
Ready.


In [37]:
# 1. Elimina por la fuerza el Pod que está corriendo LocalStack por error
!kubectl delete pod architect-python-batch-job-29718976-vxqvb --force



pod "architect-python-batch-job-29718976-vxqvb" force deleted


In [38]:
# 2. Re-aplica el archivo asegurando que limpie los trabajos viejos
!kubectl apply -f cronjob.yaml

cronjob.batch/architect-python-batch-job unchanged


In [39]:
!kubectl get pods

NAME                                        READY   STATUS    RESTARTS   AGE
architect-python-batch-job-29718976-hjxp5   1/1     Running   0          54s


In [40]:
!kubectl logs architect-python-batch-job-29718976-hjxp5


LocalStack version: 2.3.2
LocalStack build date: 2023-10-03
LocalStack build git hash: 1bccf790

2026-07-04T04:35:08.463  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T04:35:08.463  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T04:35:08.694  INFO --- [  MainThread] localstack.utils.bootstrap : Execution of "start_runtime_components" took 1205.53ms
Ready.


In [43]:
!kubectl run batch-test-direct --image=python:3.11-slim --restart=Never --env="AWS_ACCESS_KEY_ID=mock_key" --env="AWS_SECRET_ACCESS_KEY=mock_secret" --env="AWS_DEFAULT_REGION=us-east-1" --env="AWS_ENDPOINT_URL=http://host.docker.internal:4566" --env="SIMULATION_MODE=OFF"

Error from server (AlreadyExists): pods "batch-test-direct" already exists


In [46]:
!kubectl logs batch-test-direct

In [47]:
!docker system prune -f 

Deleted Containers:
7056991b19ff41f9c3a84813a1b22d84c3403d9c6190242dee2b6dc37f0bc759
45013c1b6b428097408a451be5b02e191d0cba879ed4fe1db9b5f541ecad2d64
bd4506f21895933e11f152b8f48769da1e1d5d033e1fa2c47494c93bf80e2116
54c66cc2e849f49e8429b81ba6c35aa451c6525c90e8bba4a5bdb6d7f91f7ffa
8c09c215594877259fd7e26ebaebc2b638eaea3caf1b9ba62332ce7dc66cf4de

Deleted Networks:
seventh_container_default

Deleted Images:
deleted: sha256:03488e350aef96e9426975c8eddbbd995d8c6f0e6e2e827d1436f4348b7085f8
deleted: sha256:07ad5bb6d8c8f44ac5259609a0ce83315d4f2957e2e3980c4407dd0c6d1599e6
deleted: sha256:ac9685610758abd0948582ec493570f360bb9fd538ff41ab6dfeeaac7d9fe407
deleted: sha256:192ea4eb14637d7f1e120c7b4b4cc1d2a094f2657fb5abdc9568dbb35abed9ce
deleted: sha256:2873f2c787053c24647c381682115c3c5e1f367e1345d17278948f0f45a09c27
deleted: sha256:bec5e95990efe5d3a4bb84e2ef628258aa90c3452df0d4b217359a43fd210a87
deleted: sha256:893f948d3849f603fde83b8ebdb892eaed5284e4a4002bc19dd9364dd7646d73
deleted: sha256:cc28095ee4

In [ ]:
Deleted Containers:
7056991b19ff41f9c3a84813a1b22d84c3403d9c6190242dee2b6dc37f0bc759
45013c1b6b428097408a451be5b02e191d0cba879ed4fe1db9b5f541ecad2d64
bd4506f21895933e11f152b8f48769da1e1d5d033e1fa2c47494c93bf80e2116
54c66cc2e849f49e8429b81ba6c35aa451c6525c90e8bba4a5bdb6d7f91f7ffa
8c09c215594877259fd7e26ebaebc2b638eaea3caf1b9ba62332ce7dc66cf4de

Deleted Networks:
seventh_container_default

Deleted Images:
deleted: sha256:03488e350aef96e9426975c8eddbbd995d8c6f0e6e2e827d1436f4348b7085f8
deleted: sha256:07ad5bb6d8c8f44ac5259609a0ce83315d4f2957e2e3980c4407dd0c6d1599e6
deleted: sha256:ac9685610758abd0948582ec493570f360bb9fd538ff41ab6dfeeaac7d9fe407
deleted: sha256:192ea4eb14637d7f1e120c7b4b4cc1d2a094f2657fb5abdc9568dbb35abed9ce
deleted: sha256:2873f2c787053c24647c381682115c3c5e1f367e1345d17278948f0f45a09c27
deleted: sha256:bec5e95990efe5d3a4bb84e2ef628258aa90c3452df0d4b217359a43fd210a87
deleted: sha256:893f948d3849f603fde83b8ebdb892eaed5284e4a4002bc19dd9364dd7646d73
deleted: sha256:cc28095ee4635701d7207aad2d5d6929c683a36b74ac9e3111c7a66285e51a99
deleted: sha256:32b6ecff315fbcc9cc97642e59bdddf25b0eb45a8151cc43dc37f3b575af8769
deleted: sha256:bd3b7653ba574ae90517efdbecab21d443494b6edd93d8af3221b944194f6428
deleted: sha256:27966439429341d982efa402b0d49e3d24d982e39ff17671c5037a81650f2849
deleted: sha256:007fbd908487bf976acbaf8d45b7698e217c27fae57793135f82b3c77dd50e34
deleted: sha256:791bb84b2b2797fbec9a39403c668e8d3091f540a6209bf504e4a87c1b85e3ca
deleted: sha256:04f467c5ddf1a1707969425ce15fd65d5366e551da1ea0f7e319b33e7de9331f
deleted: sha256:feef2fb5d596696ce15368f5bbcd26c84150a2dd4d2a2e0659e96e1da51a4187
deleted: sha256:61ef49087c1f9869a1d1e262d1ce71d898d7a73efe00a79a2d29799be25d5d38
deleted: sha256:2ca6a990c0a5e6a0ab37ad64745ce475d2ac1eed958386ab753854ec8d4471d7
deleted: sha256:6748e2afcc587638b23c69ab17361e11cf05721515a1747d6e9cf39f3b11d3b6
deleted: sha256:a93dad551460ee28101716b0df708310f06776f1cf668608ee4354255563f751
deleted: sha256:ad1928a368a495739178c0f2ac1702fe4c8ed86aef43bd777cca80f80520b015
deleted: sha256:4b100ea43e10333517b3ca8a013f15a9966fc547e00b0aa53cd90878cba6ce52
deleted: sha256:69faad2b9071233dfdeef6ee10b9fd14660b4c140a517a2a909e837f3459f1e5
deleted: sha256:54cd79a54387dbe19720883cc7596afbd51a443c987aaee96faa56cee145e616
deleted: sha256:ec5e5a0345fe1a60734422e3b496f76286d6385784d39d455cd02012fc29420b
deleted: sha256:476ebe54f3dd585fc2cf352d5ec02dc7cc2dab323d65fb1e52fb849f24160e74
deleted: sha256:66320f7797f99f146113aee8a764b29a11d3ac2665e407f3cfc9be48ba1cab84
deleted: sha256:5a6171930d4b77b7763e4762e592ac7f4f8849c6898cbe835431a9e085ebfa7e
deleted: sha256:3652db0b53899bd41cae6c00fea8d9cb24b6760c039d1f249caff826e34c095b
deleted: sha256:b66986480ef40a1c475a51b538028e50bdd8b30f8b2c20edf060b71018151f47
deleted: sha256:95cf23c4183f62146c1513b8f0eb146a6e4e26a01314d113e7bf1b3248533503
deleted: sha256:4b9607dd73c96222627eb7900aa491d0d33e1e1a851d54a99557ec05accdd03d
deleted: sha256:18eeedbc80a529a0739f2cf807d8d6504ac19a626359ff0b2fa4b703668801db
deleted: sha256:fae24f9a9ccf4e796e17751454386d68a4072ab23537ab8c3a079297d406500d
deleted: sha256:2a8783e0c177272a20d677c77ea5931e8dedd204dfdfbf63b8ec9968cbbd4f00
deleted: sha256:a482407e1cdbcf527ec2739e28bb44620c8a001f12dd5c09948b6d63169dcc58
deleted: sha256:8aa8a700850cfbf8ead4bcf1dd7f3a9e9f5f97ca22df38e943591b9d95fd2e81
deleted: sha256:80229c7fc1c94f34204fab15cd8a461df1614d420ea14f3693ca4704be3069d1
deleted: sha256:21a2963f575c2cbf1651c20bf60689df09efb5d24c7806fdf50560d8700f103f
deleted: sha256:45efc415d701721de68ddebee3eb4baf758914c829446fbb02971fa338043ad8
deleted: sha256:bd9dcf92e26f122c87a59fa8727bed3d0d20f3c9c1ea6c4aa88955eac9c14e5b
deleted: sha256:ca59396fb26579032fce2af37949a837e103c2a450740900e2d187ac816071a8
deleted: sha256:9d7500555913639486857f4dcf22006b9cff44715758aedb4da7e0ebed43c860
deleted: sha256:e00ffe0c07a8195108a08bd44de8db4baf02d7ff87df49a7842dfbf7869e817b
deleted: sha256:cdfa2975e339095e8baea7cc4fe6626e3aaaafea4a330d174c3de53db62e742f
deleted: sha256:75dac8ba781ee15976485a996270b40cc7c1a35966ca01498ee365507a27d2d7
deleted: sha256:c2d75338f257793344e2a3a690e26a69a592009919c13e49ffd72c162cc1a2c8
deleted: sha256:b267ca26fad77af67e242dcc1f0b35da8e1dfd868821d26f26bea75d87fb74f8
deleted: sha256:3884b84fca7b68bbc087f3abfb9defda15af5eebafce6f6c65c88c220322ca41
deleted: sha256:b9684b3a1e00564dd10eff3af84b6db4e76c2a04631820a5502ab66b395330f9
deleted: sha256:9b86fe07533e22a0184361af24ade26226e1b35c62e2dc325520780403cbf3f3
deleted: sha256:b02cc36f8afb283684dea4745cbe705ed5168f36c68e6c948f2d70b72f885a71
deleted: sha256:36c3807418e935c0a26cbfb994cbb35ab0fd9322c00caaa09f6270f9f297abdf
deleted: sha256:a87b827b9c041ae42e01901c2e152a695d08a5c29180ea40dbcb1599bd836c52
deleted: sha256:5d0d2b0993eb22a205ade04250cbc089d6c723914fc16a3df69ecbb829377ec8
deleted: sha256:09de85df8d5eda1246ae2069783a8020a602dfb27dcbe6e2d2b6bb2913f844fc
deleted: sha256:9ef6c085966afe3a00a2369c49be4ba445f8d629fe443579e95cb2437454272b
deleted: sha256:9c7ee2c00d22f4b517edb8ca1c7575bbb4939fd9a39584a2673ff06c7546c291
deleted: sha256:a73aea4554dce077af07d14dc8ae0e8531214a3c2ca01219213a5ea66f9221bf
deleted: sha256:b83b7ce99367a658d624b73a62ee0555b49c5ef189967f1b939ae7b30ee0a15e
deleted: sha256:07736627e0852f4096d940866891a03ca0982043881809c8dcced16e5fa4124c
deleted: sha256:f8969dea507d3ccb6f9e9d41638e934d839dc383146a2f071a4b553086f1faeb
deleted: sha256:33e54547831675fb75b4b050af1bf0b401283e1f6e638b06f720fd1c96a1695d
deleted: sha256:c3d6a3f8a63610ef3285a71e88e6e87e9e218d2e322d02c7d32a950e8c43228e
deleted: sha256:d73d5cf649b49b930884ea87c1ba28995cda6a0c42a26665cc7b9558d3264914
deleted: sha256:b4721fe24806878b5da02794c2a990270117abde025bdfc038c4c4d421822fe4
deleted: sha256:152a18743e9492fc8dda6401755a53e5cbd2a96c893e39222aeae141d5a515ca
deleted: sha256:206d87b312eeecd966fef179cbdfe3069f57b2179490f664b3ff347266651583
deleted: sha256:8b54962a26406936386732752c9edd8d8db27bdc6d0c98153f4bb02bcaa76936
deleted: sha256:35e17cb348801971d3f097cc0aec678a1507ccd6dd3d3f3ae88e10877fb8b266
deleted: sha256:24eadda9a37304de12fb2d7ea7feae55caabb26963d283ebe33b609704118964
deleted: sha256:58611b56f1767eef921c99dae7823c3dbfaf37eace0adaf7ff1ea0e390ae1a48
deleted: sha256:5321530bd7906b5c0ca5decf577211fc205a2d6004fe9ed3c46223c40db7bb8e
deleted: sha256:f5cac482ae3552271e52cc40ee37133bbad679f47b75fa09ad483aee7b8a8bd4
deleted: sha256:acbd80bebdc5e92cc0be1e09c9ea8a00a572d80aad832f7e073b8624e9410b8e

Deleted build cache objects:
hmbi7gr79n3b3g2e6dn8yvnho
wdqugspm2qaui4cnde3a7qr3r
pv4ehjr1xuhktn8exoa4thsof
cmeu80f29exckp3jn4n8o5rdo
dl1udjsd2i2n5eqdo6kqzr4zb
iqhe7a6j09hzgkcx4nyu0bh4e
wuow3t0isqa3gx7tm3evz23a4
s1clmeyd3yo8qi54019aw0229
m86zqysjuectn91kco2col5lk
rgei7zf1orhwvu9yhb1toi4wq
2r5tawo51muoi2r35g6w4cqjj
4z8dvwm0a2sgd7rjh20cr3bqh
9b1b555d216073tybt49a3ocl
lgv3hc8pi0fr67zg7gfrxb65l
5vxa1ibpx6z6t3h8txivfg60o
sifn09cv30j6wnwrkca77bowc
j4clpdqrkssn837fhufw8ibr9
hdp1c2sx5x9v9qh9w2kpdrlhf
xuzzt6qna7b7o722o5ju4rpat
qggh3vza3glrkv5uaqkxgsim0
lpe223jmfa8v7t5y66a7axr7l
zoit0opp9gxq0to8hovhg94sf
5smlnxvf09gt77ezw8nzq77a9
dycacj6pp3udffr69hl535l9j
9xgsioaj503rh89d0tafxhjit
iesiu0736ii41l0xgfdlrmb6a
qzk17e7rj5abjl1z5q5uro8v0
81qulo8zq5lqrkifr5mh2pxv7
lpf7hwmfs19sv8m4r4c77wzwq
aha92wh6s0gul3gfct2xyke2i
t4vsd7nkq9j3bgctid5prvrxt
cau8og9ttb2tzk2qvinpxvfxp
ny8m8f73mo5i6wqno8kczh7af
i436w5oq9ew9mtttg7ciw45v0
t3oh7iyznuwfloqjnng70xgq6
uboqogq33wxppoy0y9xwdt0sp
03nwlqfv5s93bdw2vsazzefxi
us2ili8qw04yq3kfxs1eclowt
byxac7goymmtxv05lq1amids8
c4h8uelkjnw3tsy61vze9t1uu
5nyy1djerftj8jrq7xajw674v
u3g6j2jxdylulgyfti6mtv48i
wy0l9nxx6ban9uifyi1x6wr13
96b64a7udykty2wkatiljedm2
99auwuh8aj6imj8g96ujeplia
f11u90dtksalyhs9dyfalztdc
337tc3vz38j2hwff5tbunm9od
q5xbh22cj3c7f2dbkldub74cc
jcgwyzw83o9uf07yhn87nfqgn
xfj0uqnjue5zlrh4pwnve1xgd
ye9b5pcxu9djiapfrp1yn22nh
q5nh0a66j661c09nbgyf54r61
oko7wgrg0k2sl6yrguqcotdr3
qi5st5md2wd32pijun6dvse2r
ru38pomhd1hzos3mkc27vc4yt
yciurfykq6kwe6cwlafxpj92b
j2fns2rena07zhz7y1uyyru2i
naarqnjso7nex86mwoyyod3ut
movvodbpwjgcgl49shm9fjkpl
j0kmj3lwjf0edlxfw5gmt1tb0
mrjwueql4qka809ofxbpcg44h
sq4q1bnqngymgm0819cbd5pi3
inmfzj7jtya9wwwuiutohup88
skskfrqsojntlgbmz70u7dt8e
0z29otwgzu0to236ax1znfpko
3sfx2g4nbc876xs92zt54r3ez
ys3innprui06h2n5vhio9pns8
w91wt4t1jp57osm21r7uiazil
ykkndojqc0pi7y6eg5go0nmhn
j9ams6mkks62mmjrxydoxkbkp
m9jq28uyxyg7dty41z7o85nkl
uohd0rvfc6yplch7a5xtrrib1
we6033wzuuua4xgz2xzrl2ddl
m7qzhjh9pqqma4nu9qxqmqogy
qpz4ng7baqob0p8x0xc5h6tga
dc9totc9hbe18q14svv658dbs
pi89fa2hrnt2uqa6gbvgpgtf0
i1peleevf4b1zoc7juk3iz52a
4j4vw65y56swhod00vj5klrxa
gkf32qbz6tc9sbberwqcsc5qs
zk7mk4olk1dqcp5ytpqaszbjn
11cc9ef4civ30hibgda4nbctc
m1orvmfjcyoqlx8k72ypd2uzh
f34sonucrplby7r2jff7etfj5
i8oh7hmotmnm3crdt4zekopd6
l273sk3h2i6ow1qx0mqrib3mf
qe20tp8x42rtgaq0rfiyxmp6d
tk46hzvan82dgjs71cte5ioof
dad96e1fuaam8o9noazk44z9h
w51msmzg7hklgtqfykimuex6e
j31bniy3p2zkfix7uwbvu83ag
liemrtx087l40um59z7yooaxk
ohpcnz6x5tz5lp9o60n1xsz7m
86q3x7r8owcz8ee66jb6i03tl
ihkinu1mkni04oot3fiugjff7
j8l3xncsose9pqv0zhtzlpi9z
lvel7d8fat9xnaicjqk1frf2l
5gwsvg1spe7sdype4hqd4hrcj
j9boy2xzsmx8qz3r5j0xkptmz
mlfcgn9ksxwxziqwpfdhwh3k1
wn4s9rhvv7t4odsk3wcqohy72
yzqzm75xwjbyjzjmpuqzuahgj
bfv2hzlm4u8v05j69kp8yvquq
nlt72b1ym24wf9y1j7vwkfq02
6m4h4m3l90wv3q470oir8h08o
dn0irsuwao0n0hi9i2de0vxmy
u4d20ifja1h40w99u0ambkylr
ae5ubz0h4j8irz5m3auxjn092
q3k0ih13kuhcunc8ruw54ja74
tlpj8c9oegom14lahv1iduptx
92cgy2akmtk0kpou7vsf2r8h7
90l6di26hf3c6gdwh4rlqijol
p8pvec9bcs53lv5pp8iuk2w7g
r4hmyyb9m78wufsxcxx5vb92u
mafp3fzhm3czyxvvbl6ruqqj6
gf8yhef2sbirk4jed0aabz1we
o7keaobegd2togpoiesg9eiox
wjy6auaq0blypohr6mcob9rz3
1ne7iury0ynitzff33rmzvb2k
jymhxjcy86xovrnlao8cv0kap
uy387sofyeylq8drm0pj92qlm
qkufrbsk7eylkfusrlbk7jdvo
hg9zfl4xl7pfxaewcrn34fm99
ejafpsxls1kjg56jzi1i98zqb
wyhdk8strfxsn8ol80u9a4ggo
vuvwr51z34tsyyw5cx6wwsx34
9gpva7cmg2oxpw4cesmpon1rb
6s5p72qj3i4nwu0j9rt0bany3
c1v7dg54tclmi0eixp007la83
rx1p913kcpqwhy0tuoeoci22r
yltc45x48oqr4tdwdxtyaaawq
vcjr567kh9mjl34jli7rmo312
uzkfku0snmvkr8xiunwkkv3lv
ivut78kke1r6plgo1kgbv2jzb
4hw92g0f2r061nvjftluze9x6
g1e0tgk3f0oos0l6v435mzlyg
mauvspvu2irj2bt44yisvhyeq
lidc4bs5m885jb413ji263hsa
v3ulxu24kciriyf50v0vbtsyi
nffd19143fzd8dli3krkoywlf
tudqygyvnamyo73pnwhkvkgna
lz3ie3xz92fvjzogxy5omraq5
vmbv5lmldyye1v25yi2ur5aou
wgua5ppcv7fdwzcqqbwluivcg
mj8egjhyqls288laabd82hww0
t6cl5v2agbm1yfk8uv7ff35r3
98h1jtfz8181r8zmrb2lc1sax
6ls3envmfgddbss89alki8ezp
pc73788d0u87wxnup6gigng7u
rz7e8zl1gktod4ivmvsdxmnss
22anhhg8e0cp0u4x1l7mm68ob
og61jr3cgfjtzbb7tk4oykb40
olt18y0a2bhp4lxz0pcent58y
1z3ipgd84iqrct2pd26utv8pz
v2i50nipzo0s69n5czzwke8yu
h06dray0lie0spn215icfoq79
mavf6eumonazxqxe2rgpyi1xe
glae3r7np3rwixpa730yjb5mq
yepteq3rwhmrg5qgx45u0cdgc
sf31ehm575gkhwy6q44m7pk7y
uzvn4sr3w1nokvk1ze70euc72
vsvxyw31xc09zkv9amw0f3dqp
z63zmhhq6gvzh6lwlmsmou81m
qzu72hjnttqm5ehw5nvomff2r
kb5juevi2fe8xoiz5te1prsnb
f7em6zjfq8h1mdemp3husmd87
s3xo72knps7qxhh4gov2xpdw0
sv6cguphxzfc87i97ypwza2s6
c7168sggks6ih3e20g0ntxwdu
xwcq8a4p7xuhveepc4vbfh9r5
my0opgddht0bjl2n4ikfvp4zm
qhjmjjv6goasmcmm90xg4bin9
6wdatpxxvfjufj62dj3m3rg6d
ik03k8d62izozrqpd45hd1w9j
qyiv93xdnhdxfmmpytrl9o8pn
s01lbl8idoap500vv0lb7e5yj
b2t2siyjoecruvddqz15qyc96
juw27hmwdnai4awzju83l76k5
rxzsrf4ltkj7fv0u457lgbtfq
qeffao5w9r9o2t04h408f130q
04wkiep82yrelrkona4w9uskf
nadj1eamf9jd7pbzglkhk6khh
252nd3kk5vyu5p6mo2id8holt
vzqlhuimg4zj0lmce5ohytjoa
hc0all8w285bdow6oyctfjnv5
j80kxj98pquv6zxzqg4gm8qdq
xh4gi3dhky1x7lxquuadsi1zn
bur9203hpwbt6cot4789av1i3

Total reclaimed space: 3.401GB

Click to add a cell.

In [1]:
!kubectl delete pod batch-test-direct --force

The connection to the server kubernetes.docker.internal:6443 was refused - did you specify the right host or port?


In [3]:
!kubectl apply -f cronjob.yaml

cronjob.batch/architect-python-batch-job unchanged


In [4]:
!kubectl get pods

NAME                                        READY   STATUS      RESTARTS        AGE
architect-python-batch-job-29718976-hjxp5   1/1     Running     1 (6m41s ago)   62m
batch-test-direct                           0/1     Completed   0               57m


In [5]:
!kubectl logs architect-python-batch-job-29718976-hjxp5 --tail=50


LocalStack version: 2.3.2
LocalStack build date: 2023-10-03
LocalStack build git hash: 1bccf790

2026-07-04T05:31:58.376  INFO --- [-functhread4] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T05:31:58.376  INFO --- [-functhread4] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T05:31:58.453  INFO --- [  MainThread] localstack.utils.bootstrap : Execution of "start_runtime_components" took 1213.58ms
Ready.


In [2]:
# 1. Mira qué deployments o cronjobs tienes activos
!kubectl get all

# 2. Si ves algo en la lista, borra TODO lo que esté corriendo en tu namespace por defecto
!kubectl delete all --all

NAME                                            READY   STATUS             RESTARTS         AGE
pod/architect-python-batch-job-29719782-v6hng   0/1     Completed          0                2m59s
pod/architect-python-batch-job-29719783-w9qd7   0/1     Completed          0                119s
pod/architect-python-batch-job-29719784-mqb4m   0/1     Completed          0                59s
pod/architect-python-run-once-dj6vr             0/1     Completed          0                10h
pod/batch-force-test                            0/1     Completed          0                11h
pod/batch-test-direct                           0/1     Completed          0                13h
pod/network-test                                0/1     CrashLoopBackOff   25 (3m10s ago)   10h

NAME                        TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)    AGE
service/kubernetes          ClusterIP   10.96.0.1       <none>        443/TCP    14h
service/localstack-bridge   ClusterIP   10.101.131.42   <n

In [1]:
!docker compose down --remove-orphans

no configuration file provided: not found


In [6]:
!kubectl delete pod architect-python-batch-job-29718976-hjxp5 --force

pod "architect-python-batch-job-29718976-hjxp5" force deleted


In [7]:
!kubectl apply -f cronjob.yaml

cronjob.batch/architect-python-batch-job unchanged


In [8]:
!kubectl get pods

NAME                                        READY   STATUS      RESTARTS   AGE
architect-python-batch-job-29718976-8rwt7   1/1     Running     0          57s
batch-test-direct                           0/1     Completed   0          63m


In [9]:
!kubectl logs architect-python-batch-job-29718976-8rwt7


LocalStack version: 2.3.2
LocalStack build date: 2023-10-03
LocalStack build git hash: 1bccf790

2026-07-04T05:43:13.822  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T05:43:13.822  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T05:43:14.139  INFO --- [  MainThread] localstack.utils.bootstrap : Execution of "start_runtime_components" took 1270.44ms
Ready.


In [11]:
yaml_content = """
apiVersion: batch/v1
kind: CronJob
metadata:
  name: architect-python-batch-job
  namespace: default
spec:
  schedule: "*/1 * * * *"
  concurrencyPolicy: Forbid
  successfulJobsHistoryLimit: 3
  failedJobsHistoryLimit: 1
  jobTemplate:
    spec:
      template:
        metadata:
          labels:
            app: telemetry-batch-processor
        spec:
          containers:
          - name: telemetry-batch-processor
            image: python:3.11-slim
            imagePullPolicy: IfNotPresent
            command: ["python", "-c", "import os; print('=== ENTORNO INICIADO CORRECTAMENTE ==='); print('AWS ID:', os.environ.get('AWS_ACCESS_KEY_ID'))"]
            env:
            - name: AWS_ACCESS_KEY_ID
              value: "mock_key"
            - name: AWS_SECRET_ACCESS_KEY
              value: "mock_secret"
            - name: AWS_DEFAULT_REGION
              value: "us-east-1"
            - name: AWS_ENDPOINT_URL
              value: "http://host.docker.internal:4566"
            - name: SIMULATION_MODE
              value: "OFF"
          restartPolicy: OnFailure
"""

with open("cronjob.yaml", "w") as f:
    f.write(yaml_content.strip())
print("¡Archivo cronjob.yaml guardado con éxito en el directorio de Jupyter!")

¡Archivo cronjob.yaml guardado con éxito en el directorio de Jupyter!


In [12]:
!kubectl delete pods -l app=telemetry-batch-processor --force
!kubectl apply -f cronjob.yaml

pod "architect-python-batch-job-29718976-8rwt7" force deleted
cronjob.batch/architect-python-batch-job configured


In [13]:
!kubectl get pods

NAME                                        READY   STATUS      RESTARTS   AGE
architect-python-batch-job-29718976-b5znc   1/1     Running     0          29s
batch-test-direct                           0/1     Completed   0          78m


In [14]:
!kubectl logs architect-python-batch-job-29718976-b5znc


LocalStack version: 2.3.2
LocalStack build date: 2023-10-03
LocalStack build git hash: 1bccf790

2026-07-04T05:58:27.368  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T05:58:27.368  INFO --- [-functhread6] hypercorn.error            : Running on https://0.0.0.0:4566 (CTRL + C to quit)
2026-07-04T05:58:27.525  INFO --- [  MainThread] localstack.utils.bootstrap : Execution of "start_runtime_components" took 1215.14ms
Ready.


In [15]:
!kubectl run batch-force-test --image=python:3.11-slim --restart=Never --command -- python -c "import os; print('=== VICTORIA ABSOLUTA DE PYTHON ==='); print('Clave:', os.environ.get('AWS_ACCESS_KEY_ID'))" --env="AWS_ACCESS_KEY_ID=mock_key_direct"

pod/batch-force-test created


In [16]:
!kubectl logs batch-force-test

=== VICTORIA ABSOLUTA DE PYTHON ===
Clave: None


In [17]:
import subprocess

# Leemos tu código completo y lo guardamos localmente por seguridad
app_code = """
import io
import os
import time
from pathlib import Path
from datetime import datetime
import boto3
from botocore.exceptions import ClientError

SIMULATION_ENV = os.environ.get("SIMULACION", "False")
SIMULATION_MODE = SIMULATION_ENV.upper() == "TRUE"

AWS_ENDPOINT = os.environ.get("AWS_ENDPOINT_URL", "http://host.docker.internal:4566").strip()
AWS_BUCKET = os.environ.get("S3_BUCKET_NAME", "fase5-storage-bucket")
AWS_KEY = os.environ.get("AWS_INPUT_KEY", "raw_telemetry.csv")
LOCK_TABLE = os.environ.get("DYNAMODB_LOCK_TABLE", "fase5-execution-locks")

print(f"🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
print(f"🎛️  Modo de Simulación: {'ACTIVADO (3 filas en memoria)' if SIMULATION_MODE else 'DESACTIVADO (1,000 filas reales en LocalStack S3)'}")

BASE_DIR = Path("/app")
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def acquire_lock(dynamodb_client, lock_id, ttl_seconds=20):
    current_time = int(time.time())
    expires_at = current_time + ttl_seconds
    try:
        dynamodb_client.put_item(
            TableName=LOCK_TABLE,
            Item={
                'LockID': {'S': lock_id},
                'Status': {'S': 'LOCKED'},
                'ExpiresAt': {'N': str(expires_at)}
            },
            ConditionExpression='attribute_not_exists(LockID) OR ExpiresAt < :now',
            ExpressionAttributeValues={':now': {'N': str(current_time)}}
        )
        print(f"🔒 [Lock] Candado adquirido con éxito para la tarea: {lock_id}")
        return True
    except ClientError as e:
        if e.response['Error']['Code'] == 'ConditionalCheckFailedException':
            print(f"⚠️ [Lock] La ejecución {lock_id} ya está bloqueada por otra instancia.")
            return False
        raise e

def release_lock(dynamodb_client, lock_id):
    try:
        dynamodb_client.delete_item(TableName=LOCK_TABLE, Key={'LockID': {'S': lock_id}})
        print(f"🔓 [Lock] Candado liberado para la tarea: {lock_id}")
    except Exception as e:
        print(f"❌ Error al liberar candado {lock_id}: {str(e)}")

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\\n"
        )
    else:
        try:
            response = s3_client.get_object(Bucket=bucket, Key=key)
            s3_data = response["Body"].read().decode("utf-8")
        except ClientError as e:
            if e.response['Error']['Code'] == 'NoSuchKey':
                print(f"⚠️ [S3] No se encontró el archivo '{key}'. Creando lote inicial de 1000 filas...")
                headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\\n"
                bulk_data = headers
                for i in range(1, 1001):
                    code = "P0300" if i % 50 == 0 else "P0171" if i % 75 == 0 else "NONE"
                    bulk_data += f"{datetime.now()},VIN-{100000 + i},2100,90,{code}\\n"
                s3_client.put_object(Bucket=bucket, Key=key, Body=bulk_data.encode("utf-8"))
                s3_data = bulk_data
            else:
                raise e
    
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda guardada en contenedor.")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def local_bulk_writer(s3_client, bucket, blob_name, processed_records, simulation_mode=False):
    if not processed_records:
        print("📥 No se encontraron alertas críticas en este lote.")
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\\n"
        )
    
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo.")
    else:
        output_key = f"processed_outputs/{blob_name}"
        s3_client.put_object(Bucket=bucket, Key=output_key, Body=csv_buffer.encode("utf-8"))
        print(f"☁️  [LocalStack] Transmisión exitosa a S3 de salida: {output_key}")
    
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en contenedor.")

if __name__ == '__main__':
    # Usamos host.docker.internal para alcanzar el LocalStack nativo de tu Docker de Mac
    s3 = boto3.client("s3", endpoint_url=AWS_ENDPOINT, region_name="us-east-1")
    dynamodb_client = boto3.client("dynamodb", endpoint_url=AWS_ENDPOINT, region_name="us-east-1")

    if not SIMULATION_MODE:
        try:
            s3.create_bucket(Bucket=AWS_BUCKET)
        except Exception:
            pass

    execution_id = f"cron_batch_{datetime.now().strftime('%Y%m%d_%H%M')}"
    print(f"⏱️ Iniciando ejecución única de Kubernetes para el bloque: {execution_id}")

    if acquire_lock(dynamodb_client, execution_id):
        try:
            print("📥 Connecting to S3 Intake Stream...")
            data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULATION_MODE)
            
            incidents = []
            for raw_row in data_stream:
                clean_row = telemetry_processor(raw_row)
                if clean_row:
                    incidents.append(clean_row)
                    
            print(f"📤 Preparing handoff for {len(incidents)} processed records...")
            local_bulk_writer(s3, AWS_BUCKET, f"alerts_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv", incidents, simulation_mode=SIMULATION_MODE)
            print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")
        finally:
            release_lock(dynamodb_client, execution_id)
    else:
        print("🛑 Finalizando ejecución sin procesar para evitar duplicidad.")
"""

# Borramos ConfigMaps viejos y creamos el nuevo directo en Kubernetes con tu código entero adentro
subprocess.run("kubectl delete configmap batch-code-config --ignore-not-found", shell=True)
process = subprocess.Popen(["kubectl", "create", "configmap", "batch-code-config", "--from-file=app.py=/dev/stdin"], stdin=subprocess.PIPE)
process.communicate(input=app_code.encode('utf-8'))
print("✅ ¡Código app.py inyectado exitosamente en el ConfigMap de Kubernetes!")

configmap/batch-code-config created
✅ ¡Código app.py inyectado exitosamente en el ConfigMap de Kubernetes!


In [18]:
!cat <<EOF | kubectl apply -f -
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  template:
    spec:
      containers:
      - name: batch-processor
        image: python:3.11-slim
        command: ["python", "/app/app.py"]
        env:
        - name: SIMULACION
          value: "False"
        - name: AWS_ACCESS_KEY_ID
          value: "mock_key"
        - name: AWS_SECRET_ACCESS_KEY
          value: "mock_secret"
        - name: AWS_DEFAULT_REGION
          value: "us-east-1"
        - name: AWS_ENDPOINT_URL
          value: "http://host.docker.internal:4566"
        volumeMounts:
        - name: code-volume
          mountPath: /app
      volumes:
      - name: code-volume
        configMap:
          name: batch-code-config
      restartPolicy: Never
EOF

SyntaxError: invalid syntax (1970983853.py, line 4)

In [19]:
import subprocess

# Definimos la estructura exacta del Job sin pasar por la terminal
job_yaml = """
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  template:
    spec:
      containers:
      - name: batch-processor
        image: python:3.11-slim
        command: ["python", "/app/app.py"]
        env:
        - name: SIMULACION
          value: "False"
        - name: AWS_ACCESS_KEY_ID
          value: "mock_key"
        - name: AWS_SECRET_ACCESS_KEY
          value: "mock_secret"
        - name: AWS_DEFAULT_REGION
          value: "us-east-1"
        - name: AWS_ENDPOINT_URL
          value: "http://host.docker.internal:4566"
        volumeMounts:
        - name: code-volume
          mountPath: /app
      volumes:
      - name: code-volume
        configMap:
          name: batch-code-config
      restartPolicy: Never
"""

# Borramos ejecuciones previas para evitar colisiones de nombres
subprocess.run("kubectl delete job architect-python-run-once --ignore-not-found", shell=True)

# Inyectamos el Job de forma nativa en el clúster
process = subprocess.Popen(["kubectl", "apply", "-f", "-"], stdin=subprocess.PIPE)
process.communicate(input=job_yaml.strip().encode('utf-8'))
print("🚀 ¡Job 'architect-python-run-once' creado con éxito!")

job.batch/architect-python-run-once created
🚀 ¡Job 'architect-python-run-once' creado con éxito!


In [20]:
!kubectl logs job/architect-python-run-once

Found 7 pods, using pod/architect-python-run-once-kbjlm
Traceback (most recent call last):
  File "/app/app.py", line 7, in <module>
    import boto3
ModuleNotFoundError: No module named 'boto3'


In [21]:
import subprocess

# Estructura del Job con instalación de dependencias en caliente
job_yaml = """
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  template:
    spec:
      containers:
      - name: batch-processor
        image: python:3.11-slim
        command: ["/bin/sh", "-c", "pip install --no-cache-dir boto3 && python /app/app.py"]
        env:
        - name: SIMULACION
          value: "False"
        - name: AWS_ACCESS_KEY_ID
          value: "mock_key"
        - name: AWS_SECRET_ACCESS_KEY
          value: "mock_secret"
        - name: AWS_DEFAULT_REGION
          value: "us-east-1"
        - name: AWS_ENDPOINT_URL
          value: "http://host.docker.internal:4566"
        volumeMounts:
        - name: code-volume
          mountPath: /app
      volumes:
      - name: code-volume
        configMap:
          name: batch-code-config
      restartPolicy: Never
"""

# Borramos la ejecución anterior para aplicar la nueva configuración
subprocess.run("kubectl delete job architect-python-run-once --ignore-not-found", shell=True)

# Inyectamos el Job actualizado en el clúster
process = subprocess.Popen(["kubectl", "apply", "-f", "-"], stdin=subprocess.PIPE)
process.communicate(input=job_yaml.strip().encode('utf-8'))
print("🚀 ¡Job actualizado e inyectado con instalación de boto3 en progreso!")

job.batch "architect-python-run-once" deleted
job.batch/architect-python-run-once created
🚀 ¡Job actualizado e inyectado con instalación de boto3 en progreso!


In [22]:
!kubectl logs job/architect-python-run-once

Found 2 pods, using pod/architect-python-run-once-q6n99
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [23]:
!kubectl logs job/architect-python-run-once

Found 4 pods, using pod/architect-python-run-once-vzc4n
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 955.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 158.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 6.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [24]:
%%bash
# Celda 3: Probar conectividad con LocalStack usando un Pod temporal nativo en Jupyter
echo "🔍 Lanzando Pod temporal de diagnóstico de red..."

kubectl run k8s-localstack-ping --rm -i --restart=Never --image=python:3.11-slim -- \
python -c "
import urllib.request, json
try:
    res = urllib.request.urlopen('http://host.docker.internal:4566/_localstack/health', timeout=5).read().decode()
    print('\n✅ CONEXIÓN EXITOSA CON LOCALSTACK:\n', json.dumps(json.loads(res), indent=2))
except Exception as e:
    print('\n❌ ERROR DE CONEXIÓN:', e)
"

🔍 Lanzando Pod temporal de diagnóstico de red...


If you don't see a command prompt, try pressing enter.



❌ ERROR DE CONEXIÓN: <urlopen error [Errno 111] Connection refused>
pod "k8s-localstack-ping" deleted


In [25]:
ports:
  - "0.0.0.0:4566:4566"

SyntaxError: invalid syntax (2686928563.py, line 1)

In [26]:
%%bash
echo "🔍 Probando con el DNS nativo de Kubernetes para Docker Desktop..."

kubectl run k8s-localstack-ping-v2 --rm -i --restart=Never --image=python:3.11-slim -- \
python -c "
import urllib.request, json
try:
    res = urllib.request.urlopen('http://kubernetes.docker.internal:4566/_localstack/health', timeout=5).read().decode()
    print('\n✅ CONEXIÓN EXITOSA CON DNS INTERNO:\n', json.dumps(json.loads(res), indent=2))
except Exception as e:
    print('\n❌ TAMPOCO CONECTÓ:', e)
"

🔍 Probando con el DNS nativo de Kubernetes para Docker Desktop...


If you don't see a command prompt, try pressing enter.


pod "k8s-localstack-ping-v2" deleted


In [27]:
%%bash
echo "🚀 Levantando Pod de prueba..."
kubectl run k8s-localstack-debug --restart=Never --image=python:3.11-slim -- \
python -c "
import urllib.request
try:
    res = urllib.request.urlopen('http://kubernetes.docker.internal:4566/_localstack/health', timeout=3).read().decode()
    print('RESULTADO: EXITOSO')
except Exception as e:
    print('RESULTADO: FALLÓ ->', e)
"

# Darle un breve momento para ejecutar
sleep 3

echo "📊 --- LOGS DEL POD ---"
kubectl logs k8s-localstack-debug

echo "🧹 --- LIMPIANDO ---"
kubectl delete pod k8s-localstack-debug --wait=false


🚀 Levantando Pod de prueba...
pod/k8s-localstack-debug created
📊 --- LOGS DEL POD ---
RESULTADO: FALLÓ -> <urlopen error [Errno 111] Connection refused>
🧹 --- LIMPIANDO ---
pod "k8s-localstack-debug" deleted


In [28]:
%%bash
# Celda nueva: Forzar un puente de red directo hacia el puerto 4566 del host
echo "🔌 Creando puente de red interno hacia LocalStack..."

# Levantamos un servicio rápido en K8s que mapee el host externo
kubectl delete service localstack-bridge --ignore-not-found

kubectl create service externalname localstack-bridge --external-name=host.docker.internal

echo "✅ Puente creado. Ahora tu código debe apuntar a: http://localstack-bridge:4566"

🔌 Creando puente de red interno hacia LocalStack...
service/localstack-bridge created
✅ Puente creado. Ahora tu código debe apuntar a: http://localstack-bridge:4566


In [29]:
import subprocess

job_yaml = """
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  template:
    spec:
      containers:
      - name: batch-processor
        image: python:3.11-slim
        command: ["/bin/sh", "-c", "pip install --no-cache-dir boto3 && python /app/app.py"]
        env:
        - name: SIMULACION
          value: "False"
        - name: AWS_ACCESS_KEY_ID
          value: "mock_key"
        - name: AWS_SECRET_ACCESS_KEY
          value: "mock_secret"
        - name: AWS_DEFAULT_REGION
          value: "us-east-1"
        - name: AWS_ENDPOINT_URL
          value: "http://localstack-bridge:4566"  # <-- CAMBIADO AQUÍ PARA USAR EL PUENTE
        volumeMounts:
        - name: code-volume
          mountPath: /app
      volumes:
      - name: code-volume
        configMap:
          name: batch-code-config
      restartPolicy: Never
"""

subprocess.run("kubectl delete job architect-python-run-once --ignore-not-found", shell=True)
process = subprocess.Popen(["kubectl", "apply", "-f", "-"], stdin=subprocess.PIPE)
process.communicate(input=job_yaml.strip().encode('utf-8'))
print("🚀 Job desplegado con éxito usando el nuevo puente de red.")

job.batch "architect-python-run-once" deleted
job.batch/architect-python-run-once created
🚀 Job desplegado con éxito usando el nuevo puente de red.


In [30]:
%%bash
# Celda nueva: Monitorear el progreso y los logs del Job
echo "📊 Buscando el Pod del Job..."
POD_NAME=$(kubectl get pods -l job-name=architect-python-run-once -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$POD_NAME" ]; then
    echo "⚠️ El Pod aún se está creando. Esperando 3 segundos..."
    sleep 3
    POD_NAME=$(kubectl get pods -l job-name=architect-python-run-once -o jsonpath='{.items[0].metadata.name}')
fi

echo "📌 Estado actual del Pod ($POD_NAME):"
kubectl get pod $POD_NAME -o jsonpath='{.status.phase}'
echo -e "\n"

echo "📜 --- CAPTURANDO LOGS ---"
kubectl logs job/architect-python-run-once --tail=50

📊 Buscando el Pod del Job...
📌 Estado actual del Pod (architect-python-run-once-pzhvw):
Running

📜 --- CAPTURANDO LOGS ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 1.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [31]:
%%bash
echo "📜 --- CAPTURANDO LOGS ACTUALIZADOS ---"
kubectl logs job/architect-python-run-once --tail=50

📜 --- CAPTURANDO LOGS ACTUALIZADOS ---


Found 2 pods, using pod/architect-python-run-once-pzhvw


  File "/usr/local/lib/python3.11/site-packages/botocore/client.py", line 1076, in _make_api_call
    http, parsed_response = self._make_request(
                            ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/botocore/client.py", line 1100, in _make_request
    return self._endpoint.make_request(operation_model, request_dict)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/botocore/endpoint.py", line 119, in make_request
    return self._send_request(request_dict, operation_model)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/botocore/endpoint.py", line 200, in _send_request
    while self._needs_retry(
          ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/botocore/endpoint.py", line 360, in _needs_retry
    responses = self._event_emitter.emit(
                ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/

In [33]:
import subprocess

app_code = """
import io
import os
import time
from pathlib import Path
from datetime import datetime
import boto3
from botocore.exceptions import ClientError

SIMULATION_ENV = os.environ.get("SIMULACION", "False")
SIMULATION_MODE = SIMULATION_ENV.upper() == "TRUE"

# `.strip()` remueve espacios y `.strip("/")` quita cualquier barra diagonal que rompa a boto3
AWS_ENDPOINT = os.environ.get("AWS_ENDPOINT_URL", "http://localstack-bridge:4566").strip().strip("/")
AWS_BUCKET = os.environ.get("S3_BUCKET_NAME", "fase5-storage-bucket")
AWS_KEY = os.environ.get("AWS_INPUT_KEY", "raw_telemetry.csv")
LOCK_TABLE = os.environ.get("DYNAMODB_LOCK_TABLE", "fase5-execution-locks")

print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
print(f"🔗 Conectando al Endpoint: {AWS_ENDPOINT}")
BASE_DIR = Path("/app")

# ... Resto de tu lógica de candados en DynamoDB e ingesta de 1,000 filas ...
"""

# Limpiar e inyectar código corregido en memoria de Kubernetes
subprocess.run("kubectl delete configmap batch-code-config --ignore-not-found", shell=True)
process = subprocess.Popen(["kubectl", "create", "configmap", "batch-code-config", "--from-file=app.py=/dev/stdin"], stdin=subprocess.PIPE)
process.communicate(input=app_code.encode('utf-8'))
print("✅ ConfigMap 'batch-code-config' actualizado y limpio.")

configmap "batch-code-config" deleted
configmap/batch-code-config created
✅ ConfigMap 'batch-code-config' actualizado y limpio.


In [34]:
import subprocess

job_yaml = """
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  template:
    spec:
      containers:
      - name: batch-processor
        image: python:3.11-slim
        command: ["/bin/sh", "-c", "pip install --no-cache-dir boto3 && python /app/app.py"]
        env:
        - name: SIMULACION
          value: "False"
        - name: AWS_ACCESS_KEY_ID
          value: "mock_key"
        - name: AWS_SECRET_ACCESS_KEY
          value: "mock_secret"
        - name: AWS_DEFAULT_REGION
          value: "us-east-1"
        - name: AWS_ENDPOINT_URL
          value: "http://localstack-bridge:4566"
        volumeMounts:
        - name: code-volume
          mountPath: /app
      volumes:
      - name: code-volume
        configMap:
          name: batch-code-config
      restartPolicy: Never
"""

subprocess.run("kubectl delete job architect-python-run-once --ignore-not-found", shell=True)
process = subprocess.Popen(["kubectl", "apply", "-f", "-"], stdin=subprocess.PIPE)
process.communicate(input=job_yaml.strip().encode('utf-8'))
print("🚀 Job redesplegado con éxito con el ConfigMap actualizado.")

job.batch "architect-python-run-once" deleted
job.batch/architect-python-run-once created
🚀 Job redesplegado con éxito con el ConfigMap actualizado.


In [35]:
%%bash
echo "📊 --- REVISANDO LOGS DE EJECUCIÓN ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- REVISANDO LOGS DE EJECUCIÓN ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 65.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
🔗 Conectando al Endpoint: http://localstack-bridge:4566


In [36]:
%%bash
echo "📊 --- PROGRESO DEL PIPELINE ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- PROGRESO DEL PIPELINE ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 65.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
🔗 Conectando al Endpoint: http://localstack-bridge:4566


In [37]:
kubectl run network-test --rm -i --tty --image=busybox -- restart=Never -- wget -T 5 -O- http://localstack-bridge:4566/health

SyntaxError: invalid syntax (277755018.py, line 1)

In [38]:
!kubectl run network-test --rm -i --tty --image=busybox -- restart=Never -- wget -T 5 -O- http://localstack-bridge:4566/health

^C


In [39]:
%%bash
kubectl get svc localstack-bridge

NAME                TYPE           CLUSTER-IP   EXTERNAL-IP            PORT(S)   AGE
localstack-bridge   ExternalName   <none>       host.docker.internal   <none>    38m


In [40]:
%%bash
kubectl delete svc localstack-bridge

service "localstack-bridge" deleted


In [41]:
%%bash
cat <<EOF | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  name: localstack-bridge
spec:
  ports:
    - protocol: TCP
      port: 4566
      targetPort: 4566
---
apiVersion: v1
kind: Endpoints
metadata:
  name: localstack-bridge
subsets:
  - addresses:
      - ip: 10.0.2.2  # IP típica del Host para Minikube. Si usas Docker Desktop en Mac/Win, a veces se usa la IP local de tu máquina (ej. 192.168.1.X)
    ports:
      - port: 4566
EOF

service/localstack-bridge created
endpoints/localstack-bridge created


In [42]:
%%bash
kubectl delete job architect-python-run-once

job.batch "architect-python-run-once" deleted


In [43]:
%%bash
# Cambia 'job.yaml' por el nombre de tu archivo si lo tienes guardado
kubectl apply -f job.yaml

error: the path "job.yaml" does not exist


CalledProcessError: Command 'b"# Cambia 'job.yaml' por el nombre de tu archivo si lo tienes guardado\nkubectl apply -f job.yaml\n"' returned non-zero exit status 1.

In [44]:
%%bash
# 1. Limpiamos el Job anterior para que no deje rastro
kubectl delete job architect-python-run-once --ignore-not-found

# 2. Creamos y lanzamos el nuevo Job en caliente
cat <<EOF | kubectl apply -f -
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: architect-python-container
          image: python:3.11-slim
          command: ["/bin/sh", "-c"]
          args:
            - |
              echo "📦 Instalando dependencias en runtime..."
              pip install --quiet boto3
              
              echo "🚀 Booting up Multi-Cloud Pipeline..."
              echo "🔗 Conectando al Endpoint: http://localstack-bridge:4566"
              
              python -c "
import boto3
from botocore.exceptions import EndpointConnectionError

try:
    s3 = boto3.client(
        's3',
        endpoint_url='http://localstack-bridge:4566',
        aws_access_key_id='test',
        aws_secret_access_key='test',
        region_name='us-east-1'
    )
    response = s3.list_buckets()
    print('✅ Conexión exitosa a LocalStack!')
    print('📊 Buckets disponibles:', response.get('Buckets', []))
except EndpointConnectionError as e:
    print('❌ Error: No se pudo conectar a LocalStack.', e)
except Exception as e:
    print('⚠️ Ocurrió otro error:', e)
              "
EOF

error: error parsing STDIN: error converting YAML to JSON: yaml: line 24: could not find expected ':'


CalledProcessError: Command 'b'# 1. Limpiamos el Job anterior para que no deje rastro\nkubectl delete job architect-python-run-once --ignore-not-found\n\n# 2. Creamos y lanzamos el nuevo Job en caliente\ncat <<EOF | kubectl apply -f -\napiVersion: batch/v1\nkind: Job\nmetadata:\n  name: architect-python-run-once\nspec:\n  backoffLimit: 1\n  template:\n    spec:\n      restartPolicy: Never\n      containers:\n        - name: architect-python-container\n          image: python:3.11-slim\n          command: ["/bin/sh", "-c"]\n          args:\n            - |\n              echo "\xf0\x9f\x93\xa6 Instalando dependencias en runtime..."\n              pip install --quiet boto3\n              \n              echo "\xf0\x9f\x9a\x80 Booting up Multi-Cloud Pipeline..."\n              echo "\xf0\x9f\x94\x97 Conectando al Endpoint: http://localstack-bridge:4566"\n              \n              python -c "\nimport boto3\nfrom botocore.exceptions import EndpointConnectionError\n\ntry:\n    s3 = boto3

In [45]:
%%writefile test_s3.py
import boto3
from botocore.exceptions import EndpointConnectionError

print("🚀 Booting up Multi-Cloud Pipeline (Fase 6 Test)...")
print("🔗 Conectando al Endpoint: http://localstack-bridge:4566")

try:
    s3 = boto3.client(
        's3',
        endpoint_url='http://localstack-bridge:4566',
        aws_access_key_id='test',
        aws_secret_access_key='test',
        region_name='us-east-1'
    )
    response = s3.list_buckets()
    print("✅ Conexión exitosa a LocalStack!")
    print("📊 Buckets disponibles:", response.get('Buckets', []))
except EndpointConnectionError as e:
    print("❌ Error: No se pudo conectar a LocalStack.", e)
except Exception as e:
    print("⚠️ Ocurrió un error inesperado:", e)

Writing test_s3.py


In [47]:
%%bash
echo "📊 --- PROGRESO DEL PIPELINE ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- PROGRESO DEL PIPELINE ---


Error from server (NotFound): jobs.batch "architect-python-run-once" not found


CalledProcessError: Command 'b'echo "\xf0\x9f\x93\x8a --- PROGRESO DEL PIPELINE ---"\nkubectl logs job/architect-python-run-once --tail=50\n'' returned non-zero exit status 1.

In [48]:
%%bash
echo "📊 --- PROGRESO DEL PIPELINE ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- PROGRESO DEL PIPELINE ---


Error from server (NotFound): jobs.batch "architect-python-run-once" not found


CalledProcessError: Command 'b'echo "\xf0\x9f\x93\x8a --- PROGRESO DEL PIPELINE ---"\nkubectl logs job/architect-python-run-once --tail=50\n'' returned non-zero exit status 1.

In [49]:
%%bash
# Borramos residuos anteriores si existen
kubectl delete configmap pipeline-script-config --ignore-not-found
kubectl delete job architect-python-run-once --ignore-not-found

# Creamos el ConfigMap con el script limpio
kubectl create configmap pipeline-script-config --from-literal=main.py='
import boto3
from botocore.exceptions import EndpointConnectionError

print("🚀 Booting up Multi-Cloud Pipeline (Fase 6)...")
print("🔗 Conectando al Endpoint: http://localstack-bridge:4566")

try:
    s3 = boto3.client(
        "s3",
        endpoint_url="http://localstack-bridge:4566",
        aws_access_key_id="test",
        aws_secret_access_key="test",
        region_name="us-east-1"
    )
    response = s3.list_buckets()
    print("✅ Conexión exitosa a LocalStack!")
    print("📊 Buckets disponibles:", response.get("Buckets", []))
except EndpointConnectionError as e:
    print("❌ Error: No se pudo conectar a LocalStack.", e)
except Exception as e:
    print("⚠️ Ocurrió un error inesperado:", e)
'

configmap/pipeline-script-config created


In [50]:
%%bash
cat <<EOF | kubectl apply -f -
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: architect-python-container
          image: python:3.11-slim
          command: ["/bin/sh", "-c"]
          args:
            - |
              echo "📦 Instalando dependencias en runtime..."
              pip install --quiet boto3
              echo "🏃 Ejecutando script de procesamiento..."
              python /script/main.py
          volumeMounts:
            - name: script-volume
              mountPath: /script
      volumes:
        - name: script-volume
          configMap:
            name: pipeline-script-config
EOF

job.batch/architect-python-run-once created


In [51]:
%%bash
echo "📊 --- PROGRESO DEL PIPELINE ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- PROGRESO DEL PIPELINE ---
📦 Instalando dependencias en runtime...


In [52]:
%%bash
echo "📊 --- PROGRESO DEL PIPELINE ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- PROGRESO DEL PIPELINE ---
📦 Instalando dependencias en runtime...

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
🏃 Ejecutando script de procesamiento...


In [53]:
%%bash
echo "📊 --- PROGRESO DEL PIPELINE ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- PROGRESO DEL PIPELINE ---
📦 Instalando dependencias en runtime...

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
🏃 Ejecutando script de procesamiento...


In [54]:
%%bash
cat <<EOF | kubectl apply -f -
apiVersion: v1
kind: Endpoints
metadata:
  name: localstack-bridge
subsets:
  - addresses:
      - ip: 192.168.65.2  # IP interna del Host para Docker Desktop
    ports:
      - port: 4566
EOF

endpoints/localstack-bridge configured


In [55]:
%%bash
# 1. Borramos el Job congelado
kubectl delete job architect-python-run-once --ignore-not-found --quiet

# 2. Creamos el Job limpio (tomando el script desde el ConfigMap que ya guardamos en Kubernetes)
cat <<EOF | kubectl apply -f -
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: architect-python-container
          image: python:3.11-slim
          command: ["/bin/sh", "-c"]
          args:
            - |
              echo "📦 Instalando dependencias en runtime..."
              pip install --quiet boto3
              echo "🏃 Ejecutando script de procesamiento..."
              python /script/main.py
          volumeMounts:
            - name: script-volume
              mountPath: /script
      volumes:
        - name: script-volume
          configMap:
            name: pipeline-script-config
EOF

error: unknown flag: --quiet
See 'kubectl delete --help' for usage.


job.batch/architect-python-run-once unchanged


In [56]:
%%bash
# 1. Borramos el Job congelado de forma limpia (sin flags inválidos)
kubectl delete job architect-python-run-once --ignore-not-found

# 2. Recreamos el Job interactivo apuntando al ConfigMap
cat <<EOF | kubectl apply -f -
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: architect-python-container
          image: python:3.11-slim
          command: ["/bin/sh", "-c"]
          args:
            - |
              echo "📦 Instalando dependencias en runtime..."
              pip install --quiet boto3
              echo "🏃 Ejecutando script de procesamiento..."
              python /script/main.py
          volumeMounts:
            - name: script-volume
              mountPath: /script
      volumes:
        - name: script-volume
          configMap:
            name: pipeline-script-config
EOF

job.batch "architect-python-run-once" deleted
job.batch/architect-python-run-once created


In [57]:
%%bash
echo "📊 --- PROGRESO DEL PIPELINE ---"
kubectl logs job/architect-python-run-once --tail=50

📊 --- PROGRESO DEL PIPELINE ---
📦 Instalando dependencias en runtime...

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
🏃 Ejecutando script de procesamiento...
🚀 Booting up Multi-Cloud Pipeline (Fase 6)...
🔗 Conectando al Endpoint: http://localstack-bridge:4566
❌ Error: No se pudo conectar a LocalStack. Could not connect to the endpoint URL: "http://localstack-bridge:4566/"


In [58]:
%%bash
kubectl delete svc localstack-bridge --ignore-not-found
kubectl delete endpoints localstack-bridge --ignore-not-found

service "localstack-bridge" deleted


In [59]:
%%bash
cat <<EOF | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  name: localstack-bridge
spec:
  ports:
    - protocol: TCP
      port: 4566
      targetPort: 4566
EOF

service/localstack-bridge created


In [60]:
%%bash
# Iniciamos el port-forward inverso hacia tu máquina local en segundo plano
kubectl port-forward service/localstack-bridge 4566:4566 > /dev/null 2>&1 &
echo "🔄 Túnel interactivo establecido con LocalStack en segundo plano."

🔄 Túnel interactivo establecido con LocalStack en segundo plano.


In [61]:
%%bash
kubectl delete job architect-python-run-once --ignore-not-found
cat <<EOF | kubectl apply -f -
apiVersion: batch/v1
kind: Job
metadata:
  name: architect-python-run-once
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: architect-python-container
          image: python:3.11-slim
          command: ["/bin/sh", "-c"]
          args:
            - |
              echo "📦 Instalando dependencias en runtime..."
              pip install --quiet boto3
              echo "🏃 Ejecutando script de procesamiento..."
              python /script/main.py
          volumeMounts:
            - name: script-volume
              mountPath: /script
      volumes:
        - name: script-volume
          configMap:
            name: pipeline-script-config
EOF

job.batch "architect-python-run-once" deleted
job.batch/architect-python-run-once created


In [62]:
!kubectl logs job/architect-python-run-once

📦 Instalando dependencias en runtime...

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
🏃 Ejecutando script de procesamiento...
